# CA4 Project 37: Hotel Revenue & Occupancy Analytics
**Business Analytics | Google Colab analysis notebook**

**Before you begin:** faculty approval is confirmed. This notebook downloads a public hotel-booking dataset, inspects data quality, performs exploratory analysis, calculates descriptive KPIs, and exports figures/tables for the report.

**Important limitations**
- The source contains booking-level records, not a full inventory of rooms available per hotel per day. Do not claim to calculate true occupancy percentage.
- ADR × length of stay is only a rough room-value proxy, not audited or necessarily realized revenue.
- This is historical data for two hotels; results should not be generalized to all hotels or current market conditions.

## 1. Install/import libraries

In [ ]:
!pip -q install pandas numpy matplotlib seaborn

In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme()
OUTPUT_DIR = Path("project_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load the public dataset

In [ ]:
DATA_URLS = [
    "https://raw.githubusercontent.com/aaqibqadeer/Hotel-booking-demand/master/hotel_bookings.csv",
    "https://raw.githubusercontent.com/Juandi22/Hotel-Booking-Demand/master/hotel_bookings.csv",
]
df_raw = None
load_errors = []
for url in DATA_URLS:
    try:
        df_raw = pd.read_csv(url)
        print(f"Loaded dataset from: {url}")
        break
    except Exception as exc:
        load_errors.append(f"{url}: {exc}")
if df_raw is None:
    raise RuntimeError(
        "Could not download the CSV. Download hotel_bookings.csv from the approved public dataset source "
        "and upload it to Colab, then use: df_raw = pd.read_csv('/content/hotel_bookings.csv')\n"
        + "\n".join(load_errors)
    )
print("Raw shape:", df_raw.shape)
display(df_raw.head())

## 3. Data overview and quality checks

In [ ]:
print("Rows, columns:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())
print("\nData types and non-null counts:")
df_raw.info()

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df_raw) * 100).round(2)
})
display(missing_table[missing_table["missing_count"] > 0])
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))

## 4. Clean and prepare data

In [ ]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

# Convert expected numeric fields without silently dropping rows.
numeric_cols = [
    "is_canceled", "lead_time", "arrival_date_year", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes", "days_in_waiting_list",
    "adr", "required_car_parking_spaces", "total_of_special_requests"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Report duplicate count; remove exact duplicate records for analysis and preserve original df_raw.
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Normalize empty strings in text fields.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

# Keep missing child values missing in the original field; use zero only for derived guest count,
# and explicitly document this assumption.
if "children" in df.columns:
    df["children_for_guest_count"] = df["children"].fillna(0)
else:
    df["children_for_guest_count"] = 0

df["adults_for_guest_count"] = df["adults"].fillna(0)
df["babies_for_guest_count"] = df["babies"].fillna(0)
df["guest_count"] = df["adults_for_guest_count"] + df["children_for_guest_count"] + df["babies_for_guest_count"]
df["total_nights"] = df["stays_in_weekend_nights"].fillna(0) + df["stays_in_week_nights"].fillna(0)
df["booking_status"] = df["is_canceled"].map({0: "Not canceled", 1: "Canceled"}).fillna("Unknown")
# Proxy only. Clip negative ADR at zero for this derived indicator; retain original adr for review.
df["estimated_room_value_proxy"] = df["adr"].clip(lower=0) * df["total_nights"]

# Month ordering for charts.
month_order = ["January","February","March","April","May","June","July","August","September","October","November","December"]
if "arrival_date_month" in df.columns:
    df["arrival_date_month"] = pd.Categorical(df["arrival_date_month"], categories=month_order, ordered=True)

print("Raw shape:", df_raw.shape)
print("Exact duplicates removed:", duplicates_before)
print("Analysis shape:", df.shape)
display(df.head())

## 5. Validate data and document cleaning

In [ ]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Raw rows", "Raw columns", "Exact duplicate rows found",
        "Rows after exact duplicate removal", "Missing values remaining (all columns)",
        "Rows with negative ADR", "Rows with zero total nights",
        "Rows with missing children"
    ],
    "Value": [
        len(df_raw), df_raw.shape[1], duplicates_before, len(df),
        int(df.isna().sum().sum()),
        int((df["adr"] < 0).sum()) if "adr" in df else np.nan,
        int((df["total_nights"] == 0).sum()),
        int(df["children"].isna().sum()) if "children" in df else np.nan
    ]
})
display(cleaning_summary)
cleaning_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)

## 6. Core KPIs

In [ ]:
completed = df[df["is_canceled"] == 0].copy()
kpis = {
    "Analysis rows after deduplication": len(df),
    "Canceled bookings": int((df["is_canceled"] == 1).sum()),
    "Not-canceled bookings": int((df["is_canceled"] == 0).sum()),
    "Cancellation rate (%)": round(df["is_canceled"].mean() * 100, 2),
    "Average ADR for not-canceled bookings": round(completed["adr"].mean(), 2),
    "Median ADR for not-canceled bookings": round(completed["adr"].median(), 2),
    "Median lead time (days)": round(df["lead_time"].median(), 2),
    "Mean length of stay (nights)": round(df["total_nights"].mean(), 2),
    "Room-value proxy for not-canceled bookings": round(completed["estimated_room_value_proxy"].sum(), 2),
}
kpi_table = pd.DataFrame({"Metric": list(kpis.keys()), "Value": list(kpis.values())})
display(kpi_table)
kpi_table.to_csv(OUTPUT_DIR / "core_kpis.csv", index=False)

## 7. Booking demand by month and hotel

In [ ]:
monthly_bookings = (df.groupby("arrival_date_month", observed=False).size()
                    .reindex(month_order).rename("Bookings").reset_index())
plt.figure(figsize=(12,5))
sns.barplot(data=monthly_bookings, x="arrival_date_month", y="Bookings")
plt.title("Booking records by arrival month")
plt.xlabel("Arrival month"); plt.ylabel("Bookings"); plt.xticks(rotation=35); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bookings_by_month.png", dpi=160)
plt.show()
display(monthly_bookings)

In [ ]:
hotel_counts = df.groupby("hotel").size().sort_values(ascending=False)
display(hotel_counts.rename("Bookings").to_frame())
plt.figure(figsize=(7,4))
sns.barplot(x=hotel_counts.index, y=hotel_counts.values)
plt.title("Booking records by hotel type")
plt.xlabel("Hotel type"); plt.ylabel("Bookings"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bookings_by_hotel.png", dpi=160)
plt.show()

## 8. ADR patterns for not-canceled bookings

In [ ]:
adr_month = (completed.groupby(["arrival_date_month", "hotel"], observed=False)["adr"]
             .mean().reset_index())
plt.figure(figsize=(12,5))
sns.lineplot(data=adr_month, x="arrival_date_month", y="adr", hue="hotel", marker="o")
plt.title("Average daily rate by month and hotel (not-canceled bookings)")
plt.xlabel("Arrival month"); plt.ylabel("ADR"); plt.xticks(rotation=35); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "adr_by_month_hotel.png", dpi=160)
plt.show()
display(adr_month.pivot(index="arrival_date_month", columns="hotel", values="adr").round(2))

## 9. Cancellation analysis

In [ ]:
cancel_hotel = (df.groupby("hotel")["is_canceled"].mean().mul(100)
                .sort_values(ascending=False).rename("Cancellation rate (%)").reset_index())
display(cancel_hotel.round(2))
plt.figure(figsize=(7,4))
sns.barplot(data=cancel_hotel, x="hotel", y="Cancellation rate (%)")
plt.title("Cancellation rate by hotel type")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cancellation_by_hotel.png", dpi=160)
plt.show()

In [ ]:
cancel_segment = (df.groupby("market_segment", dropna=False)["is_canceled"].mean().mul(100)
                  .sort_values(ascending=False).rename("Cancellation rate (%)").reset_index())
display(cancel_segment.round(2))
plt.figure(figsize=(10,5))
sns.barplot(data=cancel_segment, x="market_segment", y="Cancellation rate (%)")
plt.title("Cancellation rate by market segment")
plt.xlabel("Market segment"); plt.ylabel("Cancellation rate (%)"); plt.xticks(rotation=35); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cancellation_by_segment.png", dpi=160)
plt.show()

In [ ]:
lead_time_status = df.groupby("booking_status", dropna=False)["lead_time"].median().rename("Median lead time (days)").reset_index()
display(lead_time_status)
plt.figure(figsize=(7,4))
sns.barplot(data=lead_time_status, x="booking_status", y="Median lead time (days)")
plt.title("Median lead time by booking status")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lead_time_by_status.png", dpi=160)
plt.show()

## 10. Distribution channels and customer types

In [ ]:
for col in ["distribution_channel", "market_segment", "customer_type"]:
    summary = df[col].fillna("Missing").value_counts().rename_axis(col).reset_index(name="Bookings")
    display(summary)
    plt.figure(figsize=(10,4))
    sns.barplot(data=summary, x=col, y="Bookings")
    plt.title(f"Booking records by {col.replace('_',' ').title()}")
    plt.xticks(rotation=35); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{col}_bookings.png", dpi=160)
    plt.show()

## 11. Data tables and business questions

In [ ]:
# Save useful summary tables for report writing.
monthly_bookings.to_csv(OUTPUT_DIR / "monthly_bookings.csv", index=False)
cancel_hotel.to_csv(OUTPUT_DIR / "cancellation_by_hotel.csv", index=False)
cancel_segment.to_csv(OUTPUT_DIR / "cancellation_by_market_segment.csv", index=False)
adr_month.to_csv(OUTPUT_DIR / "adr_by_month_hotel.csv", index=False)

print("Business questions addressed:")
for q in [
    "How does booking demand vary by month and hotel type?",
    "How does ADR vary across months and hotel types for not-canceled bookings?",
    "How do cancellation rates vary by hotel type and market segment?",
    "Which distribution channels, market segments and customer types account for the most bookings?",
    "What additional fields are required to calculate actual occupancy and realized revenue?"
]:
    print("•", q)

## 12. Findings draft generator (verify before reporting)

In [ ]:
# These are descriptive patterns computed from this dataset, not causal conclusions.
# Review the displayed values and plots before using any statement in your final report.
if not monthly_bookings["Bookings"].dropna().empty:
    peak = monthly_bookings.loc[monthly_bookings["Bookings"].idxmax()]
    low = monthly_bookings.loc[monthly_bookings["Bookings"].idxmin()]
    print(f"Booking volume: highest month in the month-aggregated view is {peak['arrival_date_month']} ({int(peak['Bookings']):,} records); "
          f"lowest is {low['arrival_date_month']} ({int(low['Bookings']):,} records).")
if not cancel_hotel.empty:
    top_cancel = cancel_hotel.iloc[0]
    print(f"Hotel type with highest cancellation rate in this analysis: {top_cancel['hotel']} ({top_cancel['Cancellation rate (%)']:.2f}%).")
if not cancel_segment.empty:
    top_seg = cancel_segment.iloc[0]
    print(f"Market segment with highest cancellation rate in this analysis: {top_seg['market_segment']} ({top_seg['Cancellation rate (%)']:.2f}%).")
print("\nBefore finalizing: validate row counts, describe the sample, avoid causal claims, and discuss the lack of room-inventory data.")

## 13. Export outputs for the report

In [ ]:
# Export cleaned dataset for personal analysis. Do not commit a large raw/cleaned CSV to GitHub unless required.
df.to_csv(OUTPUT_DIR / "hotel_bookings_cleaned.csv", index=False)
print("Output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")

## 14. Final report checklist
- [ ] Replace all report placeholders with measured results from this notebook.
- [ ] Include the actual raw/cleaned row counts, missing-value checks and duplicate handling.
- [ ] Insert genuine screenshots of charts and the Streamlit dashboard.
- [ ] Tie each recommendation to a numerical finding.
- [ ] Clearly explain that true occupancy cannot be calculated without rooms-available/inventory data.
- [ ] Call ADR × nights an estimated proxy, not verified revenue.
- [ ] Add the actual GitHub repository and deployed Streamlit URLs.
- [ ] Cite the original research article and public dataset source.

## 15. Download analysis outputs
Run this final cell after the notebook has completed. It packages the tables, figures and cleaned data into one ZIP file for the report and further review.

In [ ]:
import shutil
from google.colab import files
output_zip = shutil.make_archive("hotel_analysis_outputs", "zip", OUTPUT_DIR)
files.download(output_zip)